# Clasificador binario de edad v2 — Menores (<18) vs Adultos (>=18)

Mejoras respecto a `train_classifier.ipynb`:

1. **Datasets diversos** — soporte para UTKFace, FairFace y face_age combinados
2. **Augmentation agresivo** — HueSaturationValue, GaussNoise, CoarseDropout para generalizar entre etnias
3. **Fine-tuning profundo** — layer3 + layer4 descongelados con learning rates discriminativas
4. **Training mejorado** — AdamW, CosineAnnealingLR, mas epochs, mayor dropout
5. **Evaluacion por etnia** — métricas desagregadas para detectar sesgos
6. **Backbone opcional** — EfficientNet-B2 como alternativa a ResNet-50

In [ ]:
import torch
print(torch.__version__)
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
import os
import re
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset, random_split
from torchvision import models
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_auc_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATASET_FACE_AGE = "C:\\Users\\diego\\Documents\\dataset\\face_age"
DATASET_UTKFACE  = "C:\\Users\\diego\\Documents\\dataset\\UTKFace"
DATASET_FAIRFACE = "C:\\Users\\diego\\Documents\\dataset\\FairFace"

PATH_MODELS  = "models_v2"
BATCH_SIZE   = 32
EPOCHS       = 30
PATIENCE     = 7
LR           = 3e-4
IMG_SIZE     = (224, 224)
DROPOUT      = 0.4
WEIGHT_DECAY = 1e-4
BACKBONE     = "resnet50"

os.makedirs(PATH_MODELS, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 1. Datasets Multi-Etnia

Soportamos tres formatos de dataset:

- **face_age**: carpetas nombradas por edad (ej. `0/`, `1/`, ..., `100/`)
- **UTKFace**: archivos con formato `age_gender_race_date.jpg` (race: 0=White, 1=Black, 2=Asian, 3=Indian, 4=Other)
- **FairFace**: CSV con columnas `file`, `age`, `race` (age como rango, ej. `0-2`, `3-9`, `10-19`, `20-29`, ...)

In [ ]:
class FaceAgeDataset(Dataset):
    """Dataset original: carpetas por edad. Etiqueta 1.0=menor, 0.0=adulto."""
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform
        self.ethnicities = []
        root = Path(root)
        if not root.exists():
            print(f"AVISO: {root} no existe, se omite.")
            return
        for folder in sorted(root.iterdir()):
            if not folder.is_dir():
                continue
            try:
                age = int(folder.name)
            except ValueError:
                continue
            label = 1.0 if age < 18 else 0.0
            for ext in ("*.png", "*.jpg", "*.jpeg"):
                for img_path in folder.glob(ext):
                    self.samples.append((str(img_path), label))
                    self.ethnicities.append("unknown")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.array(Image.open(path).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, torch.tensor(label, dtype=torch.float32)


class UTKFaceDataset(Dataset):
    """UTKFace: filename format age_gender_race_date.jpg
    Race: 0=White, 1=Black, 2=Asian, 3=Indian, 4=Others
    """
    RACE_MAP = {0: "White", 1: "Black", 2: "Asian", 3: "Indian", 4: "Other"}

    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform
        self.ethnicities = []
        root = Path(root)
        if not root.exists():
            print(f"AVISO: {root} no existe, se omite.")
            return
        pattern = re.compile(r"^(\d+)_(\d+)_(\d+)_")
        for img_path in root.glob("*.jpg"):
            m = pattern.match(img_path.name)
            if not m:
                continue
            age = int(m.group(1))
            race = int(m.group(3))
            if age > 116:
                continue
            label = 1.0 if age < 18 else 0.0
            self.samples.append((str(img_path), label))
            self.ethnicities.append(self.RACE_MAP.get(race, "Other"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.array(Image.open(path).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, torch.tensor(label, dtype=torch.float32)


class FairFaceDataset(Dataset):
    """FairFace: CSV-based dataset with age ranges and race labels."""
    AGE_TO_MINOR = {
        "0-2": True, "3-9": True, "10-19": True,
        "20-29": False, "30-39": False, "40-49": False,
        "50-59": False, "60-69": False, "more than 70": False,
    }

    def __init__(self, root, split="train", transform=None):
        import pandas as pd
        self.samples = []
        self.transform = transform
        self.ethnicities = []
        root = Path(root)
        csv_path = root / f"fairface_label_{split}.csv"
        if not csv_path.exists():
            print(f"AVISO: {csv_path} no existe, se omite.")
            return
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            age_group = row["age"]
            if age_group == "10-19":
                label = 1.0
            elif age_group in ("0-2", "3-9"):
                label = 1.0
            else:
                label = 0.0
            img_path = root / row["file"]
            if img_path.exists():
                self.samples.append((str(img_path), label))
                self.ethnicities.append(row.get("race", "unknown"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.array(Image.open(path).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, torch.tensor(label, dtype=torch.float32)

In [ ]:
base_tf = A.Compose([
    A.Resize(*IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

ds_face_age = FaceAgeDataset(DATASET_FACE_AGE, transform=base_tf)
ds_utk      = UTKFaceDataset(DATASET_UTKFACE, transform=base_tf)
ds_fairface = FairFaceDataset(DATASET_FAIRFACE, split="train", transform=base_tf)

print(f"face_age : {len(ds_face_age):,} imagenes")
print(f"UTKFace  : {len(ds_utk):,} imagenes")
print(f"FairFace : {len(ds_fairface):,} imagenes")

all_samples = []
all_ethnicities = []
all_labels = []

for ds in [ds_face_age, ds_utk, ds_fairface]:
    if len(ds) > 0:
        all_samples.extend(ds.samples)
        all_ethnicities.extend(ds.ethnicities)
        all_labels.extend([s[1] for s in ds.samples])

n_minor = sum(1 for l in all_labels if l == 1.0)
n_adult = sum(1 for l in all_labels if l == 0.0)
total = len(all_labels)
print(f"\nTotal combinado: {total:,}")
print(f"  Menores (<18) : {n_minor:,} ({100*n_minor/total:.1f}%)")
print(f"  Adultos (>=18): {n_adult:,} ({100*n_adult/total:.1f}%)")

eth_counts = defaultdict(int)
for e in all_ethnicities:
    eth_counts[e] += 1
print(f"\nDistribucion por etnia:")
for k, v in sorted(eth_counts.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,} ({100*v/total:.1f}%)")

## 2. Augmentation Agresivo

Augmentations clave para generalización cross-etnia:
- **HueSaturationValue**: simula variación de tono de piel
- **GaussNoise + GaussianBlur**: robustez a calidad de imagen
- **CoarseDropout**: fuerza al modelo a usar toda la cara, no solo rasgos aislados

In [ ]:
train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, border_mode=0, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.4),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CoarseDropout(max_holes=4, max_height=32, max_width=32, fill_value=0, p=0.3),
    A.Resize(*IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(*IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
viz_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, border_mode=0, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20, p=0.4),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CoarseDropout(max_holes=4, max_height=32, max_width=32, fill_value=0, p=0.3),
    A.Resize(*IMG_SIZE),
])

sample_ds = None
for ds in [ds_utk, ds_face_age, ds_fairface]:
    if len(ds) > 0:
        sample_ds = ds
        break

if sample_ds and len(sample_ds) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for i, ax in enumerate(axes.flat):
        idx = random.randint(0, len(sample_ds.samples) - 1)
        path, label = sample_ds.samples[idx]
        img = np.array(Image.open(path).convert("RGB"))
        augmented = viz_tf(image=img)["image"]
        ax.imshow(augmented)
        lbl = "Menor" if label == 1.0 else "Adulto"
        ax.set_title(lbl, fontsize=9)
        ax.axis("off")
    plt.suptitle("Ejemplos con augmentation v2")
    plt.tight_layout()
    plt.show()
else:
    print("No hay datasets disponibles para visualizar.")

## 3. Dataset Combinado con Split Train/Val

In [ ]:
class CombinedFaceAgeDataset(Dataset):
    """Combina multiples fuentes de datos con transform configurable."""
    def __init__(self, samples, ethnicities, transform=None):
        self.samples = samples
        self.ethnicities = ethnicities
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.array(Image.open(path).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, torch.tensor(label, dtype=torch.float32)


indices = list(range(total))
random.shuffle(indices)
split_idx = int(0.8 * total)

train_indices = indices[:split_idx]
val_indices   = indices[split_idx:]

train_samples     = [all_samples[i] for i in train_indices]
train_ethnicities = [all_ethnicities[i] for i in train_indices]
val_samples       = [all_samples[i] for i in val_indices]
val_ethnicities   = [all_ethnicities[i] for i in val_indices]

train_ds = CombinedFaceAgeDataset(train_samples, train_ethnicities, transform=train_tf)
val_ds   = CombinedFaceAgeDataset(val_samples, val_ethnicities, transform=val_tf)

print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}")

train_labels = [s[1] for s in train_samples]
n_minor_train = sum(1 for l in train_labels if l == 1.0)
n_adult_train = sum(1 for l in train_labels if l == 0.0)
print(f"Train — Menores: {n_minor_train:,}  Adultos: {n_adult_train:,}")

In [ ]:
weights_per_class = {1.0: 1.0 / max(n_minor_train, 1), 0.0: 1.0 / max(n_adult_train, 1)}
sample_weights = [weights_per_class[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Batches train: {len(train_loader)}  val: {len(val_loader)}")

## 4. Modelo con Fine-Tuning Profundo

- Descongelamos `layer3` + `layer4` (no solo `layer4`)
- Learning rates discriminativas: capas bajas aprenden lento, cabeza aprende rapido
- Dropout aumentado a 0.4
- Opcion de usar EfficientNet-B2 como backbone alternativo

In [ ]:
def get_resnet50_model(dropout=DROPOUT):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer3.parameters():
        param.requires_grad = True
    for param in model.layer4.parameters():
        param.requires_grad = True
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(dropout * 0.5),
        nn.Linear(256, 1),
    )
    return model


def get_efficientnet_b2_model(dropout=DROPOUT):
    model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.features[6:].parameters():
        param.requires_grad = True
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(dropout * 0.5),
        nn.Linear(256, 1),
    )
    return model


def build_model(backbone=BACKBONE):
    if backbone == "efficientnet_b2":
        return get_efficientnet_b2_model()
    return get_resnet50_model()


model = build_model(BACKBONE).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Backbone: {BACKBONE}")
print(f"Parametros totales: {total_params:,}")
print(f"Parametros entrenables: {trainable:,} ({100*trainable/total_params:.1f}%)")

## 5. Entrenamiento con AdamW + CosineAnnealingLR

In [ ]:
def binary_accuracy(logits, labels):
    preds = (logits > 0).float()
    return (preds == labels).float().mean().item()


def train_model(model, train_loader, val_loader, epochs, patience,
                lr, save_path, pos_weight_tensor=None, backbone=BACKBONE):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    if backbone == "resnet50":
        optimizer = torch.optim.AdamW([
            {"params": model.layer3.parameters(), "lr": lr * 0.1},
            {"params": model.layer4.parameters(), "lr": lr * 0.5},
            {"params": model.fc.parameters(),     "lr": lr},
        ], weight_decay=WEIGHT_DECAY)
    elif backbone == "efficientnet_b2":
        optimizer = torch.optim.AdamW([
            {"params": model.features[6:].parameters(), "lr": lr * 0.2},
            {"params": model.classifier.parameters(),   "lr": lr},
        ], weight_decay=WEIGHT_DECAY)
    else:
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=WEIGHT_DECAY
        )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_val_loss = float("inf")
    epochs_no_improve = 0
    n_train = len(train_loader.dataset)
    n_val = len(val_loader.dataset)

    for epoch in range(1, epochs + 1):
        model.train()
        t_loss, t_acc = 0.0, 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs).squeeze(1)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * len(imgs)
            t_acc += binary_accuracy(logits.detach(), labels) * len(imgs)

        model.eval()
        v_loss, v_acc = 0.0, 0.0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs).squeeze(1)
                v_loss += criterion(logits, labels).item() * len(imgs)
                v_acc += binary_accuracy(logits, labels) * len(imgs)

        train_loss = t_loss / n_train
        train_acc = t_acc / n_train
        val_loss = v_loss / n_val
        val_acc = v_acc / n_val

        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        current_lr = optimizer.param_groups[-1]["lr"]
        print(f"Epoca {epoch:2d}/{epochs}  lr={current_lr:.6f}  "
              f"train loss: {train_loss:.4f}  acc: {train_acc:.3f}  |  "
              f"val loss: {val_loss:.4f}  acc: {val_acc:.3f}")

        if val_acc > best_val_acc or (val_acc == best_val_acc and val_loss < best_val_loss):
            best_val_acc = val_acc
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print(f"  -> Mejor modelo guardado (val acc={val_acc:.4f}  loss={val_loss:.4f})")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping en epoca {epoch}")
            break

    print(f"\nMejor val accuracy: {best_val_acc:.4f}  |  Modelo: {save_path}")
    return history, best_val_acc

In [ ]:
pos_weight = torch.tensor([n_adult_train / max(n_minor_train, 1)], dtype=torch.float32).to(device)
print(f"pos_weight (compensacion desbalance): {pos_weight.item():.4f}")

save_path = os.path.join(PATH_MODELS, f"age_{BACKBONE}_v2.pth")

history, best_acc = train_model(
    model, train_loader, val_loader,
    epochs=EPOCHS, patience=PATIENCE, lr=LR,
    save_path=save_path,
    pos_weight_tensor=pos_weight,
    backbone=BACKBONE,
)

In [ ]:
def plot_history(history, title=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs_range = range(len(history["train_acc"]))

    ax1.plot(epochs_range, history["train_acc"], label="Train")
    ax1.plot(epochs_range, history["val_acc"], label="Validacion")
    ax1.set_title(f"Accuracy {title}")
    ax1.set_xlabel("Epoca")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_range, history["train_loss"], label="Train")
    ax2.plot(epochs_range, history["val_loss"], label="Validacion")
    ax2.set_title(f"Loss {title}")
    ax2.set_xlabel("Epoca")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history, f"({BACKBONE} v2)")

## 6. Evaluación Global + Por Etnia

In [ ]:
model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
model.eval()

all_preds = []
all_labels_eval = []
all_scores_eval = []
all_eth_eval = []

with torch.no_grad():
    for i in range(len(val_ds)):
        img, label = val_ds[i]
        logit = model(img.unsqueeze(0).to(device)).squeeze().item()
        score = torch.sigmoid(torch.tensor(logit)).item()
        pred = 1.0 if score > 0.5 else 0.0
        all_preds.append(pred)
        all_labels_eval.append(label.item())
        all_scores_eval.append(score)
        all_eth_eval.append(val_ethnicities[i])

all_preds = np.array(all_preds)
all_labels_eval = np.array(all_labels_eval)
all_scores_eval = np.array(all_scores_eval)

auc = roc_auc_score(all_labels_eval, all_scores_eval)
print(f"=== Evaluacion Global ===")
print(f"AUC-ROC: {auc:.4f}")
print()
print(classification_report(all_labels_eval, all_preds, target_names=["Adulto (>=18)", "Menor (<18)"]))

cm = confusion_matrix(all_labels_eval, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Adulto", "Menor"])
ax.set_yticklabels(["Adulto", "Menor"])
ax.set_xlabel("Prediccion")
ax.set_ylabel("Real")
ax.set_title("Matriz de Confusion")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
print("=== Evaluacion Por Etnia ===")
print()

ethnicities_unique = sorted(set(all_eth_eval))
results_by_eth = []

for eth in ethnicities_unique:
    mask = np.array([e == eth for e in all_eth_eval])
    if mask.sum() < 10:
        continue
    eth_labels = all_labels_eval[mask]
    eth_preds = all_preds[mask]
    eth_scores = all_scores_eval[mask]

    acc = (eth_labels == eth_preds).mean()
    try:
        eth_auc = roc_auc_score(eth_labels, eth_scores)
    except ValueError:
        eth_auc = float("nan")

    n_total_eth = mask.sum()
    n_minor_eth = (eth_labels == 1.0).sum()
    n_adult_eth = (eth_labels == 0.0).sum()

    adult_mask = eth_labels == 0.0
    adult_acc = (eth_preds[adult_mask] == 0.0).mean() if adult_mask.sum() > 0 else float("nan")

    minor_mask = eth_labels == 1.0
    minor_acc = (eth_preds[minor_mask] == 1.0).mean() if minor_mask.sum() > 0 else float("nan")

    results_by_eth.append({
        "ethnicity": eth, "n": n_total_eth, "accuracy": acc,
        "auc": eth_auc, "adult_acc": adult_acc, "minor_acc": minor_acc,
    })
    print(f"{eth:12s}  n={n_total_eth:5d}  acc={acc:.4f}  AUC={eth_auc:.4f}  "
          f"adult_acc={adult_acc:.4f}  minor_acc={minor_acc:.4f}")

print()
print("Si 'adult_acc' es baja para alguna etnia, el modelo clasifica adultos como menores (falso positivo).")
print("Esto es exactamente el problema que estamos solucionando con datos diversos.")

## 7. Exportar Modelo Final

In [ ]:
final_path = "age_model.pth"
torch.save(model.state_dict(), final_path)
print(f"Modelo final exportado a: {final_path}")
print(f"Best val accuracy: {best_acc:.4f}")
print()
print("Para usarlo en el servicio DE, copia age_model.pth a services/de/age_model.pth")
print("NOTA: Si cambiaste el backbone a efficientnet_b2, actualiza load_model() en main.py")

## 8. Test Rapido de Inferencia

In [ ]:
from torchvision import transforms as T

inference_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def predict_image(path, model, transform=inference_tf):
    img = Image.open(path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logit = model(tensor).squeeze().item()
    score = torch.sigmoid(torch.tensor(logit)).item()
    es_menor = score > 0.5
    return score, es_menor


test_dir = input("Ruta a carpeta con imagenes de test (o Enter para omitir): ").strip()
if test_dir and Path(test_dir).exists():
    for img_path in sorted(Path(test_dir).glob("*.*"))[:20]:
        if img_path.suffix.lower() in (".jpg", ".jpeg", ".png"):
            score, es_menor = predict_image(str(img_path), model)
            label = "MENOR" if es_menor else "ADULTO"
            print(f"{img_path.name:30s} → score={score:.4f}  {label}")
else:
    print("Test omitido.")